# Context Engineering

In [ ]:
import os, json
from dotenv import load_dotenv
from openai import OpenAI

# Load env and create client
load_dotenv()
API_KEY = os.getenv("OPENROUTER_API_KEY")
if not API_KEY:
    raise RuntimeError("OPENROUTER_API_KEY is required to run this notebook")
client = OpenAI(base_url="https://openrouter.ai/api/v1", api_key=API_KEY)
MODEL = "openai/gpt-oss-120b"


def call_model(messages, temperature=0):
    resp = client.chat.completions.create(
        model=MODEL, messages=messages, temperature=temperature
    )
    return resp.choices[0].message.content


def parse_strict_json(s, expected_keys, allowed_values=None, max_title_words=None):
    data = json.loads(s.strip())
    if set(data) != set(expected_keys):
        raise ValueError(f"expected exactly keys {expected_keys}, got {list(data)}")
    for key, allowed in (allowed_values or {}).items():
        if data.get(key) not in allowed:
            raise ValueError(f"invalid {key}: {data.get(key)!r}")
    if (
        max_title_words is not None
        and len(data.get("title", "").split()) > max_title_words
    ):
        raise ValueError(f"title is longer than {max_title_words} words")
    return data


def run_task(
    texts,
    developer_prompt,
    user_prompt_template,
    expected_keys,
    allowed_values=None,
    max_title_words=None,
):
    for text in texts:
        messages = [
            {"role": "developer", "content": developer_prompt},
            {"role": "user", "content": user_prompt_template.format(text=text)},
        ]
        out = call_model(messages)
        data = parse_strict_json(
            out,
            expected_keys=expected_keys,
            allowed_values=allowed_values,
            max_title_words=max_title_words,
        )
        print(json.dumps(data, ensure_ascii=False))

## 1. LLM для аналізу тексту (персона + тональність)

In [2]:
# Task 1: Strict prompt for person + tone extraction

texts = [
    "Оксана Дрозд натхненно провела демо — команда аплодувала стоячи.",
    "Керівник відділу Андрій Король грубо порушив інструкції, через що реліз зірвано.",
    "Роман Бондаренко надіслав звіт о 14:00 і підтвердив графік зустрічей.",
    "Сьогодні Ілля Ткачук блискуче оптимізував код і заощадив 30% бюджету.",
    "Марко Савченко ігнорує повідомлення й затримує погодження — це неприйнятно.",
]


PROMPT_PERSON_TONE = """
You are a deterministic JSON extractor for Ukrainian workplace messages.
Return ONLY one valid JSON object. Do not add markdown, code fences, comments, explanations, or text before/after JSON.
The object must contain EXACTLY these two keys in this meaning: "person" and "tone".
- "person": the full human name exactly as it appears in the text; remove titles such as керівник відділу.
- "tone": choose exactly one of ["позитивна", "нейтральна", "негативна"] according to the statement about that person.
Tone rules: praise, success, applause, savings, brilliant work -> "позитивна"; factual action without praise/blame -> "нейтральна"; blame, failure, blocking, ignoring, unacceptable behavior -> "негативна".
Always return exactly: {"person":"...","tone":"..."}
"""


run_task(
    texts=texts,
    developer_prompt=PROMPT_PERSON_TONE,
    user_prompt_template="Проаналізуй текст і поверни ТІЛЬКИ JSON з ключами person та tone.\nТекст: <<<{text}>>>",
    expected_keys=["person", "tone"],
    allowed_values={"tone": {"позитивна", "нейтральна", "негативна"}},
)

{"person": "Оксана Дрозд", "tone": "позитивна"}
{"person": "Андрій Король", "tone": "негативна"}
{"person": "Роман Бондаренко", "tone": "нейтральна"}
{"person": "Ілля Ткачук", "tone": "позитивна"}
{"person": "Марко Савченко", "tone": "негативна"}


## 2. Тріаж відгуків — «продукт» + «пріоритет»

In [3]:
# Task 2: triage (product + priority)

triage_texts = [
    "Квитанції не завантажуються в Atlas Pro з мобільного — фінвідділ стоїть.",
    "Після оновлення 3.1 у Core графіки будуться на 2 секунди довше, але працюють стабільно.",
    "Автовідповідь у Desk на свята — nice-to-have, допоможе трохи зменшити навантаження на операторів.",
    "Core падає під час імпорту CSV >100 тис. рядків — роботу заблоковано.",
    "Atlas показує старі курси валют, поки що оновлюємо вручну.",
]

PROMPT_PRODUCT_PRIORITY = """
You are a deterministic JSON triage extractor for Ukrainian product feedback.
Return ONLY one valid JSON object. Do not add markdown, code fences, comments, explanations, or text before/after JSON.
The object must contain EXACTLY two keys: "product" and "priority".
- "product": choose exactly one of ["Atlas", "Core", "Desk"]. Normalize variants: "Atlas Pro" -> "Atlas".
- "priority": choose exactly one of ["високий", "середній", "низький"] by business impact.
Priority rules:
* "високий": work is blocked/stopped, crashes, critical data is wrong or stale, manual workaround is required for important operations.
* "середній": feature works but slower/degraded; inconvenience without blocking work.
* "низький": nice-to-have, optional improvement, small future benefit.
Examples: "фінвідділ стоїть" -> високий; "роботу заблоковано" -> високий; "старі курси валют, оновлюємо вручну" -> високий; "на 2 секунди довше, але працюють стабільно" -> середній; "nice-to-have" -> низький.
Always return exactly: {"product":"Atlas|Core|Desk","priority":"високий|середній|низький"}
"""

run_task(
    texts=triage_texts,
    developer_prompt=PROMPT_PRODUCT_PRIORITY,
    user_prompt_template="Витягни продукт (Atlas|Core|Desk) та пріоритет (високий|середній|низький) у форматі JSON.\nТекст: <<<{text}>>>",
    expected_keys=["product", "priority"],
    allowed_values={
        "product": {"Atlas", "Core", "Desk"},
        "priority": {"високий", "середній", "низький"},
    },
)

{"product": "Atlas", "priority": "високий"}
{"product": "Core", "priority": "середній"}
{"product": "Desk", "priority": "низький"}
{"product": "Core", "priority": "високий"}
{"product": "Atlas", "priority": "високий"}


## 3. Заголовок + категорія оголошення

In [4]:
# Task 3: title + category

announce_texts = [
    "Вийшла версія 2.0 з офлайн-режимом і новим дизайном.",
    "У частини користувачів не відкривається профіль після апдейту; працюємо над виправленням.",
    "Збираємо відгуки про новий модуль аналітики.",
    "На вихідних сервіс у режимі планового обслуговування.",
    "Рекомендуємо вмикати двофакторну автентифікацію.",
]

PROMPT_TITLE_CATEGORY = """
You are a deterministic JSON generator for Ukrainian product announcements.
Return ONLY one valid JSON object. Do not add markdown, code fences, comments, explanations, or text before/after JSON.
The object must contain EXACTLY two keys: "title" and "category".
- "title": a concise Ukrainian headline, maximum 8 words, no prefix such as "Оголошення:".
- "category": choose exactly one of ["реліз", "інцидент", "рекомендація"].
Category rules:
* "реліз": new version, launch, new functionality, design update.
* "інцидент": outage, bug, degraded service, maintenance, fix in progress.
* "рекомендація": advice, request for feedback, security recommendation, best practice.
Examples: version 2.0/offline/new design -> реліз; profile does not open/fix in progress -> інцидент; feedback collection -> рекомендація; planned maintenance -> інцидент; enable 2FA -> рекомендація.
Always return exactly: {"title":"...","category":"реліз|інцидент|рекомендація"}
"""

run_task(
    texts=announce_texts,
    developer_prompt=PROMPT_TITLE_CATEGORY,
    user_prompt_template="Згенеруй короткий заголовок (≤8 слів) і категорію (реліз|інцидент|рекомендація) у форматі JSON.\nТекст: <<<{text}>>>",
    expected_keys=["title", "category"],
    allowed_values={"category": {"реліз", "інцидент", "рекомендація"}},
    max_title_words=8,
)

{"title": "Версія 2.0: офлайн‑режим і новий дизайн", "category": "реліз"}
{"title": "Профіль не відкривається після оновлення", "category": "інцидент"}
{"title": "Збираємо відгуки про новий модуль аналітики", "category": "рекомендація"}
{"title": "Сервіс у режимі планового обслуговування", "category": "інцидент"}
{"title": "Увімкніть двофакторну автентифікацію", "category": "рекомендація"}
